In [ ]:
import os, random, glob
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
N_SLICES = 9
IMAGE_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 2                 # Increase after the end-to-end check succeeds
NUM_WORKERS = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything()

train_csvs = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert train_csvs, 'Attach the competition dataset in the Notebook Input panel.'
DATA_DIR = Path(train_csvs[0]).parent
print(f'DATA_DIR: {DATA_DIR}')
print('Device:', DEVICE)

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
train_series = pd.read_csv(DATA_DIR / 'train_series.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
test_series = pd.read_csv(DATA_DIR / 'test_series.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

ID_COL = 'StudyInstanceUID'
LABELS = [c for c in sample_sub.columns if c != ID_COL]
assert set(LABELS).issubset(train.columns)
print('train / test:', train.shape, test.shape)
print('labels:', LABELS)
print('fully labelled studies:', train[LABELS].notna().all(axis=1).sum())
display(train.head(2))
display(train[LABELS].mean().sort_values().to_frame('prevalence'))
display(train_series[['Anatomical_Plane', 'Fluid_Sensitive', 'Fat_Suppression']].value_counts().head(12))

In [ ]:
def preferred_series(series_df):
    s = series_df.copy()
    s['_rank'] = (s['Anatomical_Plane'].eq('Sagittal').astype(int) * 4
                  + s['Fluid_Sensitive'].fillna(0).astype(int) * 2
                  + s['Fat_Suppression'].fillna(0).astype(int))
    return s.sort_values([ID_COL, '_rank', 'SeriesInstanceUID'], ascending=[True, False, True]).drop_duplicates(ID_COL)

train_pick = preferred_series(train_series)[[ID_COL, 'SeriesInstanceUID']]
test_pick = preferred_series(test_series)[[ID_COL, 'SeriesInstanceUID']]
train_labeled = train.loc[train[LABELS].notna().all(axis=1)].merge(train_pick, on=ID_COL, how='inner').reset_index(drop=True)
test_data = test[[ID_COL]].merge(test_pick, on=ID_COL, how='left').reset_index(drop=True)
assert len(test_data) == len(test), 'Some test studies have no selected series.'
print('trainable studies:', len(train_labeled), '| test studies:', len(test_data))

In [ ]:
def read_dicom_slices(root, study_uid, series_uid, n_slices=N_SLICES):
    files = sorted((root / str(study_uid) / str(series_uid)).glob('*.dcm'))
    if not files:
        raise FileNotFoundError(f'No DICOMs for {study_uid}/{series_uid}')
    take = np.linspace(0, len(files) - 1, n_slices).round().astype(int)
    images = []
    for i in take:
        x = pydicom.dcmread(files[i]).pixel_array.astype(np.float32)
        lo, hi = np.percentile(x, (1, 99))
        x = np.clip((x - lo) / max(hi - lo, 1e-6), 0, 1)
        x = torch.from_numpy(x)[None, None]
        x = nn.functional.interpolate(x, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)[0, 0]
        images.append(x)
    return torch.stack(images).float()

class KneeDataset(Dataset):
    def __init__(self, frame, root, labels=None):
        self.frame, self.root, self.labels = frame.reset_index(drop=True), Path(root), labels
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        x = read_dicom_slices(self.root, row[ID_COL], row.SeriesInstanceUID)
        if self.labels is None: return x, row[ID_COL]
        y = torch.tensor(row[self.labels].to_numpy(dtype=np.float32))
        return x, y

x, y = KneeDataset(train_labeled.iloc[:1], DATA_DIR / 'train_series', LABELS)[0]
print('decoded tensor:', tuple(x.shape), '| target:', y.tolist())

In [ ]:
any_positive = train_labeled[LABELS].max(axis=1)
tr_idx, va_idx = train_test_split(np.arange(len(train_labeled)), test_size=0.15, random_state=SEED, stratify=any_positive)
tr_df, va_df = train_labeled.iloc[tr_idx], train_labeled.iloc[va_idx]
train_loader = DataLoader(KneeDataset(tr_df, DATA_DIR / 'train_series', LABELS), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
valid_loader = DataLoader(KneeDataset(va_df, DATA_DIR / 'train_series', LABELS), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

class Small2p5DNet(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(in_ch, 48, 5, stride=2, padding=2), nn.BatchNorm2d(48), nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(48, 96, 3, stride=2, padding=1), nn.BatchNorm2d(96), nn.SiLU(),
            nn.Conv2d(96, 192, 3, stride=2, padding=1), nn.BatchNorm2d(192), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1))
        self.head = nn.Linear(192, out_ch)
    def forward(self, x): return self.head(self.backbone(x).flatten(1))

model = Small2p5DNet(N_SLICES, len(LABELS)).to(DEVICE)
pos = tr_df[LABELS].sum().to_numpy()
pos_weight = torch.tensor((len(tr_df) - pos) / np.maximum(pos, 1), dtype=torch.float32, device=DEVICE).clamp(max=20)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
print('positive-class weights:', dict(zip(LABELS, pos_weight.cpu().numpy().round(1))))

In [ ]:
def predict(loader):
    model.eval(); out = []
    with torch.no_grad():
        for x, _ in loader:
            out.append(torch.sigmoid(model(x.to(DEVICE, dtype=torch.float32))).cpu().numpy())
    return np.concatenate(out)

best_auc, best_state = -np.inf, None
for epoch in range(EPOCHS):
    model.train(); losses = []
    for x, y in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x.to(DEVICE, dtype=torch.float32)), y.to(DEVICE, dtype=torch.float32))
        loss.backward(); optimizer.step(); losses.append(loss.item())
    p = predict(valid_loader)
    y = va_df[LABELS].to_numpy()
    per_label = [roc_auc_score(y[:, i], p[:, i]) if len(np.unique(y[:, i])) == 2 else np.nan for i in range(len(LABELS))]
    score = np.nanmean(per_label)
    print(f'epoch {epoch + 1}/{EPOCHS}  loss={np.mean(losses):.4f}  macro_auc={score:.4f}')
    if score > best_auc:
        best_auc, best_state = score, {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
pd.DataFrame({'label': LABELS, 'auc': per_label}).sort_values('auc', ascending=False)

In [ ]:
test_loader = DataLoader(KneeDataset(test_data, DATA_DIR / 'test_series'), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_pred = predict(test_loader)
submission = pd.DataFrame(test_pred, columns=LABELS)
submission.insert(0, ID_COL, test_data[ID_COL].values)
submission.to_csv('submission.csv', index=False)
assert submission.columns.tolist() == sample_sub.columns.tolist()
assert len(submission) == len(test) and ((submission[LABELS] >= 0) & (submission[LABELS] <= 1)).all().all()
display(submission.head())
print('Wrote submission.csv:', submission.shape)